# Lp(a) Reddit Topic Modeling: original submission driver

Reproduces the tables and figures of the original submission from saved model artifacts. Written for Google Colab with a GPU runtime.

## Setup

In [ ]:
!pip install bertopic
!pip install llama-cpp-python
!pip install datamapplot
!pip install langdetect
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
import torch
import pandas as pd
import time
import numpy as np
import click
import json
import umap
import hdbscan
import bertopic
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.representation import ZeroShotClassification
from bertopic.representation import OpenAI
from bertopic.representation import LiteLLM
from bertopic.representation import KeyBERTInspired, LlamaCPP
from bertopic.representation import LangChain
from huggingface_hub import hf_hub_download
import os
from llama_cpp import Llama
import importlib
import datamapplot

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Package import

In [ ]:
import sys, importlib
# Directory containing the lpa_topic_modeling package
V3_PATH = '/path/to/Lp-a-Reddit-Topic-Modeling'
if V3_PATH not in sys.path:
    sys.path.insert(0, V3_PATH)
import lpa_topic_modeling
importlib.reload(lpa_topic_modeling)
from lpa_topic_modeling import (TopicModeler, GroupLabeler, TopicClusterer,
                                fig1a_temporal, fig1b_clusters, fig_silhouette_scan, config)

## Paths
Set `V3_PATH` (previous cell) and `BASE` to local or Drive folders before running.

In [ ]:
import os
BASE = '/path/to/project_dir'
DATA      = f'{BASE}/filtered_df_3.25.26.json'
METADATA  = f'{BASE}/metadata_3.28.26.json'
MODEL     = f'{BASE}/model_3.28.26.bertopic.pkl'
GROUPS    = f'{BASE}/group_summaries_3.30.26.json'
OUT = f'{BASE}/v3_out'; os.makedirs(OUT, exist_ok=True)
OUT_METADATA = f'{OUT}/metadata_reduced.pkl'
OUT_MODEL    = f'{OUT}/model_reduced.bertopic'
OUT_GROUPS   = f'{OUT}/group_summaries_reduced.json'

In [ ]:
reddit_df = pd.read_json(DATA)
print(len(reddit_df), 'discussions')

## Reduce outliers
One-time step that produced the reduced model loaded below; kept for reference.

In [ ]:
#modeler = TopicModeler(enable_llm=False, seed=config.SEED)
#modeler.load_complete_state(METADATA, MODEL)
# reassign outliers (deterministic); a lower threshold leaves fewer outliers
#modeler.reduce_outliers(strategy='c-tf-idf', threshold=0.1)
#modeler.save_complete_state(OUT_METADATA, OUT_MODEL)

## Load the trained model and group labels

In [ ]:
modeler = TopicModeler(enable_llm=False, seed=config.SEED)
modeler.load_complete_state(OUT_METADATA, OUT_MODEL)

In [ ]:
# one-time step: reuse the LLM group labels and refresh group sizes after outlier reduction
#labeler = GroupLabeler(modeler, preloaded_state=GROUPS)
#labeler.refresh_group_counts()
#labeler.save_group_summaries(OUT_GROUPS)
#for g, lab in labeler.current_labels().items():
#    print(g, '->', lab)

In [ ]:
labeler = GroupLabeler(modeler, preloaded_state=OUT_GROUPS)

## Figures

In [ ]:
fig1a_temporal(reddit_df, out_path=f'{OUT}/fig1a_temporal.png')
fig1b, emb2d, labels = fig1b_clusters(modeler, labeler, out_path=f'{OUT}/fig1b_clusters.png')

In [ ]:
clusterer = TopicClusterer(modeler, auto_group=False)
fig_silhouette_scan(clusterer, k_min=2, k_max=20, out_path=f'{OUT}/fig_silhouette_scan.png')

## Table 1

In [ ]:
import pandas as pd

counts = pd.Series(modeler.topics).value_counts()              # reduction-aware doc count per topic
info   = modeler.topic_model.get_topic_info()[["Topic", "Name"]]
total  = int(counts.sum())
t2g    = labeler.topic_to_group                                # {topic: 'Group_x'}
gname  = labeler.current_labels()                              # {'Group_x': 'Label'}

# topic -> readable label: prefer LLM topic labels from group_summaries, else BERTopic keywords
topic_label = {}
for s in labeler.group_summaries.values():
    if isinstance(s, dict) and s.get("topics") and s.get("topic_labels") \
       and len(s["topics"]) == len(s["topic_labels"]):
        topic_label.update({int(t): l for t, l in zip(s["topics"], s["topic_labels"])})
for _, r in info.iterrows():
    topic_label.setdefault(int(r["Topic"]), r["Name"])

td = pd.DataFrame([{"topic": t, "n": int(n), "group": gname.get(t2g.get(t), t2g.get(t))}
                   for t, n in counts.items() if t != -1])
if td["group"].isna().any():
    print("WARNING: topics with no group mapping:", td[td["group"].isna()]["topic"].tolist())

table1 = (td.groupby("group")
            .apply(lambda d: pd.Series({
                "n": int(d["n"].sum()),
                "# topics": d["topic"].nunique(),
                "Representative topics": "; ".join(
                    topic_label.get(t, str(t))
                    for t in d.sort_values("n", ascending=False)["topic"].head(3))}))
            .reset_index().rename(columns={"group": "Group"}))
table1["%"] = (100 * table1["n"] / total).round(1)
table1 = table1[["Group", "n", "%", "# topics", "Representative topics"]] \
            .sort_values("n", ascending=False).reset_index(drop=True)

n_out = int(counts.get(-1, 0))
print(f"Total {total} | categorized {total - n_out} | residual outliers {n_out} "
      f"({100*n_out/total:.1f}%) | groups {len(table1)}")
table1.to_csv(f"{OUT}/table1_groups.csv", index=False)
table1

## Table 2

In [ ]:
import pandas as pd
df = reddit_df.copy()
df["year"] = pd.to_datetime(pd.to_numeric(df["created_utc"], errors="coerce"), unit="s", utc=True).dt.year

t2 = df.groupby(["year", "type_of_entry"]).size().unstack(fill_value=0)
for c in ("post", "comment"):
    if c not in t2.columns: t2[c] = 0
t2 = t2[["post", "comment"]]
t2["total"] = t2.sum(axis=1)
t2.loc["TOTAL"] = t2.sum()

print(f'N = {len(df)} | posts {int(df.type_of_entry.eq("post").sum())} | '
      f'comments {int(df.type_of_entry.eq("comment").sum())} | '
      f'span {int(df.year.min())}–{int(df.year.max())}')
t2.to_csv(f"{OUT}/table2_by_year.csv")
t2

## Supplemental Table S1: topics

In [ ]:
import pandas as pd

counts = pd.Series(modeler.topics).value_counts()          # post-reduction doc count per topic
total  = int(counts.sum())
info   = modeler.topic_model.get_topic_info().set_index("Topic")
t2g    = labeler.topic_to_group                            # {topic: 'Group_x'}
gname  = labeler.current_labels()                          # {'Group_x': 'Label'}

# topic -> readable label: prefer LLM topic labels in group_summaries, else BERTopic keyword name
topic_label = {}
for s in labeler.group_summaries.values():
    if isinstance(s, dict) and s.get("topics") and s.get("topic_labels") \
       and len(s["topics"]) == len(s["topic_labels"]):
        topic_label.update({int(t): l for t, l in zip(s["topics"], s["topic_labels"])})

def keywords(t, k=10):
    try:    return ", ".join(w for w, _ in modeler.topic_model.get_topic(t)[:k] if w)
    except Exception: return ""

def exemplar(t, n=240):
    try:
        docs = modeler.topic_model.get_representative_docs(t)
        return (docs[0][:n] + "…") if docs else ""
    except Exception: return ""

rows = []
for t in sorted(x for x in counts.index if x != -1):
    g = t2g.get(t)
    nm = info.loc[t, "Name"] if t in info.index else str(t)
    rows.append({
        "Topic": int(t),
        "Label": topic_label.get(int(t), nm),
        "Parent group": gname.get(g, g),
        "n": int(counts.get(t, 0)),
        "%": round(100 * int(counts.get(t, 0)) / total, 2),
        "Top keywords": keywords(t),
        "Exemplar discussion": exemplar(t),
    })

suppS1 = (pd.DataFrame(rows)
            .sort_values(["Parent group", "n"], ascending=[True, False])
            .reset_index(drop=True))

n_out = int(counts.get(-1, 0))
print(f"{len(suppS1)} topics | total {total} | residual outliers {n_out} ({100*n_out/total:.1f}%)")
suppS1.to_csv(f"{OUT}/suppS1_topics.csv", index=False)
suppS1

## Supplemental figure: subreddit distribution

In [ ]:
import matplotlib.pyplot as plt, numpy as np
d = reddit_df.dropna(subset=["subreddit"]); N = len(d); U = d["subreddit"].nunique()
tot = d["subreddit"].value_counts(); top = tot.head(20).index[::-1]
pc = (d[d.subreddit.isin(top)].groupby(["subreddit","type_of_entry"]).size()
        .unstack(fill_value=0).reindex(top))
for c in ("post","comment"):
    if c not in pc: pc[c] = 0
fig, ax = plt.subplots(figsize=(8.2, 7.4)); y = np.arange(len(top))
ax.barh(y, pc["comment"], color="#6BAED6", label="comment", edgecolor="white", linewidth=0.4)
ax.barh(y, pc["post"], left=pc["comment"], color="#08519C", label="post", edgecolor="white", linewidth=0.4)
for i, s in enumerate(top):
    n = int(tot[s]); ax.text(n + N*0.004, i, f"{n:,} ({100*n/N:.1f}%)", va="center", fontsize=8)
ax.set_yticks(y); ax.set_yticklabels(["r/"+s for s in top], fontsize=9)
ax.set_xlabel("Number of discussions (posts + comments)")
ax.set_title("Distribution of Lp(a) discussions across subreddits (top 20)", loc="left", fontweight="bold")
ax.legend(title="Content type", loc="lower right"); ax.set_xlim(0, int(tot[top].max())*1.18)
for sp in ("top","right"): ax.spines[sp].set_visible(False)
ax.grid(axis="x", alpha=0.25)
ax.text(0.99, -0.085, f"{U:,} unique subreddits; top 20 account for {100*tot.head(20).sum()/N:.1f}% of {N:,} discussions.",
        transform=ax.transAxes, ha="right", va="top", fontsize=7.5, color="#555555")
fig.tight_layout(); fig.savefig(f"{OUT}/SuppFig_subreddits.png", dpi=300, bbox_inches="tight")
fig.savefig(f"{OUT}/SuppFig_subreddits.pdf", bbox_inches="tight")
print(f"{U} subreddits | top20 {100*tot.head(20).sum()/N:.1f}%")

## Supplemental table: representative excerpts
Short verbatim excerpts per keyword theme, written to a CSV for paraphrasing before publication.

In [ ]:
import pandas as pd, re
df = pd.read_json("filtered_df_3.25.26.json")
txt = (df.get("title","").fillna("").astype(str)+" "+df.get("selftext","").fillna("").astype(str)+" "+df.get("body","").fillna("").astype(str))
low = txt.str.lower(); sub = df.get("subreddit","").fillna("").astype(str)
LPA = r"lp\s*\(a\)|lipoprotein\s*\(?a\)?|\blp-?a\b"
has_lpa = low.str.contains(LPA, regex=True)

def theme(pat, extra=None, k=8, lo=45, hi=235, need_q=False):
    m = has_lpa & low.str.contains(pat, regex=True)
    if extra is not None: m = m & low.str.contains(extra, regex=True)
    if need_q: m = m & low.str.contains(r"\?", regex=True)
    out, seen = [], set()
    for i in df.index[m]:
        s = re.sub(r"\s+", " ", txt.loc[i]).strip()
        if not (lo <= len(s) <= hi): continue
        key = s[:60].lower()
        if key in seen: continue
        seen.add(key); out.append((("r/"+sub.loc[i]) if sub.loc[i] else "", s))
        if len(out) >= k: break
    return out

THEMES = {
 "Cost / access / coverage": (r"not covered|out[- ]of[- ]pocket|pay(?:ing)? for|afford|copay|co-pay|deductible|\$\s?\d|charged|reimburs|\binsurance\b|\bmedicare\b|covered by", None, False),
 "Interpreting a result (uncertainty)": (r"is (?:this|it|that|my|an?) .{0,25}(?:bad|high|normal|concerning|serious|ok|dangerous)|what does .{0,20}mean|should i (?:be )?worr|can (?:someone|anyone) explain|do i need to worry|is (?:that|this) (?:good|bad|normal)", None, True),
 "Marker discordance (Lp(a) vs LDL/apoB)": (r"apob|apo b|\bldl\b", r"despite|even though|but my|but i|however|only my|normal|low|elevated", False),
 "Family / genetic risk & screening": (r"\bfamily\b|siblings?|\bkids\b|children|relatives?|parents?|father|mother|brother|sister|\bson\b|daughter|hereditary|inherited|genetic", r"test|screen|check", False),
}
rows = []
for th, (pat, extra, need_q) in THEMES.items():
    for srd, s in theme(pat, extra, need_q=need_q):
        rows.append({"Theme": th, "Subreddit": srd, "Verbatim snippet": s, "Paraphrased for publication": ""})
pd.DataFrame(rows).to_csv("SuppTable_representative_quotes.csv", index=False)

## Figure 1B without legend
Square scatter panel and a separate legend strip with matching colors.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

_TOL = ["#4477AA","#EE6677","#228833","#CCBB44","#66CCEE",
        "#AA3377","#BBBBBB","#EE8866","#77AADD","#999933"]

def fig1b_nolegend(emb2d, labels, out="Fig1B_nolegend",
                   figsize=(7.2, 7.2), s=7, alpha=0.6):
    """Square UMAP scatter colored by group, without a legend.

    Returns:
        dict mapping each group label to its color, for fig1b_legend_only.
    """
    emb2d = np.asarray(emb2d); labels = np.asarray(labels)
    uniq = sorted(set(labels))
    colors = {g: _TOL[i % len(_TOL)] for i, g in enumerate(uniq)}
    fig, ax = plt.subplots(figsize=figsize)
    for g in uniq:
        m = labels == g
        ax.scatter(emb2d[m, 0], emb2d[m, 1], c=[colors[g]], s=s,
                   alpha=alpha, linewidths=0, rasterized=True)
    ax.set_xlabel("UMAP-1", fontsize=11); ax.set_ylabel("UMAP-2", fontsize=11)
    ax.set_xticks([]); ax.set_yticks([]); ax.margins(0.02)
    for sp in ("top", "right"): ax.spines[sp].set_visible(False)
    fig.tight_layout()
    for ext in ("pdf", "png", "svg"):
        fig.savefig(f"{out}.{ext}", dpi=300, bbox_inches="tight")
    return colors

In [ ]:
def fig1b_legend_only(colors, out="Fig1B_legend",
                      ncol=2, figsize=(8.5, 2.0)):
    """Standalone legend strip with full group names, colored to match fig1b_nolegend."""
    handles = [Line2D([0], [0], marker='o', linestyle='', markersize=10,
                      markerfacecolor=c, markeredgecolor='none', label=g)
               for g, c in colors.items()]
    fig = plt.figure(figsize=figsize)
    fig.legend(handles=handles, loc='center', ncol=ncol, frameon=False,
               fontsize=10, title="Topic groups", title_fontsize=11,
               handletextpad=0.5, columnspacing=1.6, labelspacing=0.9)
    for ext in ("pdf", "png", "svg"):
        fig.savefig(f"{out}.{ext}", dpi=300, bbox_inches="tight")

In [ ]:
colors = fig1b_nolegend(emb2d, labels, out=f'{OUT}/Fig1B_nolegend')   # square panel, no legend
fig1b_legend_only(colors, out=f'{OUT}/Fig1B_legend')                   # matching legend strip